In [3]:
from dotenv import load_dotenv
load_dotenv()

True

In [4]:
from openai import OpenAI
openai_client = OpenAI()

In [5]:
def llm(prompt):
    response = openai_client.responses.create(
        model="gpt-5.4-mini",
        input=prompt
    )
    return response.output_text

In [6]:
question="How to join this course"
answer = llm(question)
print(answer)

Sure — here’s a simple way to ask it more naturally:

**“How can I join this course?”**

If you want, I can also help you make it sound:
- **more polite**
- **more formal**
- **better for email or chat**


In [7]:
context="""
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we're still accepting submissions.

Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

What is the video/zoom link to the stream for the "Office Hours" or live/workshop sessions?
The zoom link is only published to instructors/presenters/TAs. Students participate via YouTube Live and submit questions to Slido.

Cloud alternatives with GPU
Check the quota and reset cycle carefully. Potential options include Google Colab, Kaggle, Databricks.
"""


In [8]:
prompt = f"""
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."

Question:
{question}

Context:
{context}
"""

In [9]:
answer = llm(prompt)
print(answer)

You’re accepted by default — you don’t need a confirmation email or special approval.

To join the course:
- You can just start learning right away.
- You can submit homework while the form is open.
- Registration is only for gauging interest and is not checked against a registered list.

If you want a certificate, make sure to submit your project while submissions are still being accepted.


In [10]:
def rag(question):
    search_results = search(question)
    user_prompt = build_prompt(question, search_results)
    return llm(user_prompt)

In [11]:
import requests

docs_url = "https://datatalks.club/faq/json/courses.json"
response = requests.get(docs_url)
courses_raw = response.json()

In [12]:
documents = []
url_prefix = "https://datatalks.club/faq"

for course in courses_raw:
    course_url = f"""{url_prefix}{course["path"]}"""

    course_response = requests.get(course_url)
    course_response.raise_for_status()
    course_data = course_response.json()

    documents.extend(course_data)

len(documents)

1208

In [13]:
documents[999]

{'id': 'e8dbc40d0c',
 'course': 'mlops-zoomcamp',
 'section': 'Module 1: Introduction',
 'question': 'X has 526 features, but expecting 525 features',
 'answer': 'Error:\n\n```\nValueError: X has 526 features, but LinearRegression is expecting 525 features as input.\n```\n\nSolution:\n\nThe `DictVectorizer` creates an initial mapping for the features (columns). When calling the `DictVectorizer` again for the validation dataset, `transform` should be used as it will ignore features that it did not see when `fit_transform` was last called. For example:\n\n```python\nX_train = dv.fit_transform(train_dict)\n\nX_test = dv.transform(test_dict)\n```'}

In [14]:
from minsearch import Index

index = Index(
    text_fields=["question", "section", "answer"],
    keyword_fields=["course"]
)

index.fit(documents)

In [15]:
def search(question, course="llm-zoomcamp"):
    boost_dict = {"question": 2.0, "section": 0.5}
    filter_dict = {"course": course}

    return index.search(
        question,
        boost_dict=boost_dict,
        filter_dict=filter_dict,
        num_results=5
    )

In [16]:
search_results = search(question)

In [17]:
search_results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '85384a18e5',
  'course': 'llm-zoomcamp',
  'section': 'Module 1: Introduction to LLMs and RAG',
  'question': 'OpenAI: Do I have to subscribe and pay for Open AI API for this course?',
  'answer': "No, you don't have to pay for this service in order to complete the course homeworks. You could use some of the free alternatives listed in the course GitHub.\n\n[llm-zoomcamp/01-intro/open-ai-alternatives.md at main · DataTalksClub/llm-zoomcamp (github.com)](https://github.com/DataTalksClub/llm-zoomcamp/blob/main/01-intro/open-ai-alternatives.md)"},
 {'id': 'c774542f32',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Server Error (500) When lo

In [18]:
INSTRUCTIONS = """
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."
"""

In [19]:
USER_PROMPT_TEMPLATE = """
Question:
{question}

Context:
{context}
"""

In [20]:
def build_context(search_results):
    lines = []
    
    for docs in search_results:
        lines.append(docs['section'])
        lines.append("Q: "+docs['question'])
        lines.append("A: "+docs['answer'])
        lines.append("")
        
    return "\n".join(lines).strip()

In [21]:
def build_prompt(question, search_results):
    context = build_context(search_results)
    prompt = USER_PROMPT_TEMPLATE.format(
        question=question,
        context=context
    )
    return prompt.strip()

In [22]:
prompt = build_prompt(question, search_results)

print(prompt)

Question:
How to join this course

Context:
General Course-Related Questions
Q: I just discovered the course. Can I still join?
A: Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.

Module 1: Introduction to LLMs and RAG
Q: OpenAI: Do I have to subscribe and pay for Open AI API for this course?
A: No, you don't have to pay for this service in order to complete the course homeworks. You could use some of the free alternatives listed in the course GitHub.

[llm-zoomcamp/01-intro/open-ai-alternatives.md at main · DataTalksClub/llm-zoomcamp (github.com)](https://github.com/DataTalksClub/llm-zoomcamp/blob/main/01-intro/open-ai-alternatives.md)

General Course-Related Questions
Q: Server Error (500) When logging in to course homework using GitHub
A: Additional error text seen:

```
Third-Party Login Failure

An error occurred while attempting to login via your third-party account.
```

The current solution is to use Google

In [30]:
def llm(instructions, user_prompt, model="gpt-5.4-mini"):
    message_history = [
        {"role": "developer", "content": instructions},
        {"role": "user", "content": user_prompt}
    ]

    response = openai_client.responses.create(
        model=model,
        input=message_history
    )

    return response.output_text

In [31]:
def rag(query, model="gpt-5.4-mini"):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    answer = llm(INSTRUCTIONS, prompt, model=model)
    return answer

In [32]:
answer = rag("I just discovered the course. Can I join now?")
print(answer)

Yes, you can still join. If you want a certificate, you need to submit your project while submissions are still being accepted.
